# 🔍 Tugas Lanjutan: Pengembangan Mini Search Engine
**Mata Kuliah:** Temu Kembali Informasi  
**Topik:** Green Economy & Pembangunan Berkelanjutan  

**Disusun Oleh:**
1. Bayu Nurcahyo (NIM: 3012310007)
2. Ari Setia Hinanda (NIM: 3012310005)

**Program Studi Teknik Informatika**  
**Universitas Internasional Semen Indonesia**

---

## 📋 Struktur Tugas
| No | Bagian | Keterangan |
|---|---|---|
| 1 | **Mini Search Engine** | Pre-processing, Inverted Index, TF-IDF, VSM, Antarmuka |
| 2a | **Analisis Bobot** | Perhitungan manual IDF dua kata kunci |
| 2b | **Analisis Efek Normalisasi** | Perbandingan dengan/tanpa Cosine Normalization |
| 2c | **Evaluasi Sistem** | Precision, Recall, F-Measure untuk 2 kueri |

---
**Pipeline:**  
`Dokumen Mentah → Cleaning → Tokenisasi → Stop-word Removal → Stemming → Inverted Index → TF-IDF → VSM → Cosine Similarity → Ranked Retrieval`

---
## 📦 BAGIAN 0: Instalasi & Import Library

In [ ]:
# Install library yang dibutuhkan
!pip install PySastrawi openpyxl --quiet
print("✅ Library berhasil diinstall!")

In [ ]:
# ═══════════════════════════════════════════════
#  IMPORT SEMUA LIBRARY YANG DIPERLUKAN
# ═══════════════════════════════════════════════
import pandas as pd
import numpy as np
import re
import math
import string
from collections import Counter, defaultdict
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Inisialisasi Sastrawi
stem_factory    = StemmerFactory()
stemmer         = stem_factory.create_stemmer()
stop_factory    = StopWordRemoverFactory()
stopwords_id    = set(stop_factory.get_stop_words())

print("✅ Semua library berhasil diimport!")
print(f"   → Jumlah stop-word Sastrawi: {len(stopwords_id)} kata")

---
## 📂 BAGIAN 1: Load & Persiapan Dataset

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  LOAD DATASET (dari file CSV yang sama dengan tugas sebelumnya)
# ═══════════════════════════════════════════════════════════════
df_raw = pd.read_csv('Tugas P3 TKI.csv', delimiter=';', encoding='utf-8-sig')

df = df_raw[['Kalimat/Data', 'Sumber Akses Link']].rename(columns={
    'Kalimat/Data': 'Kalimat',
    'Sumber Akses Link': 'sumber'
})
df = df.dropna(subset=['Kalimat']).reset_index(drop=True)
df['sumber'] = df['sumber'].str.extract(r'(\d+)').astype(float)

N = len(df)  # Jumlah total dokumen

print(f"✅ Dataset berhasil dimuat!")
print(f"   → Jumlah dokumen (N) : {N}")
print(f"   → Jumlah sumber      : {int(df['sumber'].max())} artikel")
df.head()

---
## 🔧 BAGIAN 2: Pre-processing & Indexing

> **Pipeline Pre-processing:**
> 1. **Text Cleaning** → Case folding, hapus angka, hapus tanda baca, normalisasi spasi
> 2. **Tokenisasi** → Pemecahan teks menjadi token/kata
> 3. **Stop-word Removal** → Hapus kata yang tidak informatif (Sastrawi)
> 4. **Stemming** → Reduksi kata ke bentuk dasar (Nazief-Adriani via PySastrawi)

In [ ]:
# ════════════════════════════════════════════════════════════════
#  STEP 1: TEXT CLEANING
#  → Case folding, hapus angka, hapus tanda baca, normalisasi spasi
# ════════════════════════════════════════════════════════════════

def clean_text(text):
    """
    Pembersihan teks (Text Cleaning):
    1. Lowercase (Case Folding) - standarisasi huruf kapital
    2. Hapus angka - angka tidak informatif untuk IR
    3. Hapus tanda baca dan karakter non-alfabet
    4. Hapus spasi berlebih (normalisasi whitespace)
    """
    text = str(text).lower()                       # 1. Case folding
    text = re.sub(r'\d+', '', text)                # 2. Hapus angka
    text = re.sub(r'[^\w\s]', ' ', text)           # 3. Hapus tanda baca
    text = re.sub(r'_', ' ', text)                 # 4. Hapus underscore
    text = re.sub(r'\s+', ' ', text).strip()       # 5. Normalisasi spasi
    return text

df['cleaned'] = df['Kalimat'].apply(clean_text)

print("=" * 70)
print(" HASIL TEXT CLEANING (Case Folding + Removal of Punctuation)")
print("=" * 70)
for i in range(3):
    print(f"\n[Kalimat {i+1}]")
    print(f"  SEBELUM : {df['Kalimat'].iloc[i][:80]}...")
    print(f"  SESUDAH : {df['cleaned'].iloc[i][:80]}...")

In [ ]:
# ════════════════════════════════════════════════════════════════
#  STEP 2: TOKENISASI
#  → Memecah teks menjadi token (list kata-kata)
# ════════════════════════════════════════════════════════════════

def tokenize(text):
    """
    Tokenisasi: memecah teks berdasarkan spasi.
    Filter token dengan panjang <= 1 karakter.
    """
    tokens = text.split()
    return [t for t in tokens if len(t) > 1]

df['tokens'] = df['cleaned'].apply(tokenize)

print("=" * 70)
print("  HASIL TOKENISASI")
print("=" * 70)
for i in range(3):
    toks = df['tokens'].iloc[i]
    print(f"\n[Kalimat {i+1}] → {len(toks)} token")
    print(f"  {toks[:10]}...")

In [ ]:
# ════════════════════════════════════════════════════════════════
#  STEP 3: STOP-WORD REMOVAL
#  → Menghapus kata-kata tidak informatif menggunakan Sastrawi
# ════════════════════════════════════════════════════════════════

def remove_stopwords(tokens, stopwords):
    """Menghapus stop-word dari list token."""
    return [t for t in tokens if t not in stopwords]

df['tokens_no_sw'] = df['tokens'].apply(lambda t: remove_stopwords(t, stopwords_id))

# Statistik
total_before = df['tokens'].apply(len).sum()
total_after  = df['tokens_no_sw'].apply(len).sum()

print("=" * 70)
print(" HASIL STOP-WORD REMOVAL")
print("=" * 70)
print(f"  Token sebelum removal : {total_before}")
print(f"  Token sesudah removal : {total_after}")
print(f"  Total token dihapus   : {total_before - total_after}")
print(f"  Persentase dikurangi  : {(total_before-total_after)/total_before*100:.1f}%")
print()
for i in range(3):
    before = df['tokens'].iloc[i]
    after  = df['tokens_no_sw'].iloc[i]
    removed = sorted(set(before) - set(after))
    print(f"[Kalimat {i+1}]")
    print(f"  Sebelum ({len(before)}) : {before[:8]}")
    print(f"  Sesudah ({len(after)})  : {after[:8]}")
    print(f"  Dihapus         : {removed}\n")

In [ ]:
# ════════════════════════════════════════════════════════════════
#  STEP 4: STEMMING (Nazief-Adriani via PySastrawi)
#  → Reduksi kata ke bentuk dasar/root word
# ════════════════════════════════════════════════════════════════

print("⏳ Proses stemming... (mungkin memakan waktu 30-60 detik)")

df['tokens_stemmed'] = df['tokens_no_sw'].apply(
    lambda tokens: [stemmer.stem(w) for w in tokens]
)

# Buat teks final (gabung kembali)
df['text_final'] = df['tokens_stemmed'].apply(lambda t: ' '.join(t))

# Kumpulkan semua token stemmed
all_stemmed = [t for doc in df['tokens_stemmed'] for t in doc]
vocabulary  = sorted(set(all_stemmed))
V           = len(vocabulary)

print("✅ Stemming selesai!")
print()
print("=" * 70)
print(" HASIL AKHIR STEMMING (5 kalimat pertama)")
print("=" * 70)
for i in range(5):
    print(f"\n[Kalimat {i+1}]")
    print(f"  Sebelum Stem : {df['tokens_no_sw'].iloc[i][:6]}")
    print(f"  Setelah Stem : {df['tokens_stemmed'].iloc[i][:6]}")

print(f"\n📊 Ukuran vocabulary akhir : {V} kata unik")

---
## 📑 BAGIAN 3: Inverted Index

> **Inverted Index** adalah struktur data inti dalam Information Retrieval.
> Setiap **term** dipetakan ke daftar **dokumen** yang mengandung term tersebut (posting list),
> beserta informasi posisi dan frekuensi kemunculan.
>
> **Struktur:** `{term: {doc_id: [positions]}}`

In [ ]:
# ════════════════════════════════════════════════════════════════
#  MEMBANGUN INVERTED INDEX
#  Struktur: { term: { 'df': int, 'postings': {doc_id: [positions]} } }
# ════════════════════════════════════════════════════════════════

def build_inverted_index(df_docs, token_col='tokens_stemmed'):
    """
    Membangun Inverted Index dari koleksi dokumen.
    
    Returns:
        inverted_index: dict dengan struktur:
          { term: { 'df': int, 'postings': {doc_id: [positions]} } }
    """
    inverted_index = defaultdict(lambda: {'df': 0, 'postings': defaultdict(list)})
    
    for doc_id, tokens in enumerate(df_docs[token_col]):
        # Catat posisi setiap token dalam dokumen
        for position, term in enumerate(tokens):
            if doc_id not in inverted_index[term]['postings']:
                inverted_index[term]['df'] += 1  # tambah DF hanya sekali per dokumen
            inverted_index[term]['postings'][doc_id].append(position)
    
    # Konversi defaultdict ke dict biasa
    return {term: {'df': data['df'], 'postings': dict(data['postings'])}
            for term, data in inverted_index.items()}

inverted_index = build_inverted_index(df)

# ── Statistik Inverted Index ──
print("╔══════════════════════════════════════════════════════════╗")
print("║          STATISTIK INVERTED INDEX                        ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Jumlah term unik (vocabulary) : {len(inverted_index):<23}║")
print(f"║  Jumlah dokumen                : {N:<23}║")
total_postings = sum(len(v['postings']) for v in inverted_index.values())
print(f"║  Total posting entries         : {total_postings:<23}║")
print("╚══════════════════════════════════════════════════════════╝")

In [ ]:
# ════════════════════════════════════════════════════════════════
#  TAMPILKAN ISI INVERTED INDEX (Contoh Term)
# ════════════════════════════════════════════════════════════════

# Urutkan berdasarkan DF (Document Frequency) tertinggi
sorted_index = sorted(inverted_index.items(), key=lambda x: x[1]['df'], reverse=True)

print("═" * 70)
print("  INVERTED INDEX — 10 TERM PALING UMUM (DF tertinggi)")
print("═" * 70)
print(f"{'Term':<20} {'DF':>5}   Posting List (Doc ID)")
print("-" * 70)

for term, data in sorted_index[:10]:
    doc_ids = sorted(data['postings'].keys())
    # Tampilkan hanya 8 doc_id pertama agar tidak terlalu panjang
    posting_str = ', '.join([f"D{d+1}" for d in doc_ids[:8]])
    if len(doc_ids) > 8:
        posting_str += f" ... (+{len(doc_ids)-8} lagi)"
    print(f"  {term:<18} {data['df']:>5}   [{posting_str}]")

print()
print("═" * 70)
print("  INVERTED INDEX — DETAIL (contoh term 'ekonomi')")
print("═" * 70)

# Detail untuk kata 'ekonomi'
if 'ekonomi' in inverted_index:
    term_data = inverted_index['ekonomi']
    print(f"  Term     : 'ekonomi'")
    print(f"  DF       : {term_data['df']} dokumen mengandung term ini")
    print(f"  Postings :")
    for doc_id, positions in list(term_data['postings'].items())[:5]:
        tf_raw = len(positions)
        print(f"    D{doc_id+1:>2}: posisi={positions[:5]}, TF={tf_raw}")

---
## ⚖️ BAGIAN 4: Pembobotan TF-IDF

> **Rumus yang digunakan:**
> - **TF (Log Frequency Weighting):** `TF(t,d) = 1 + log₁₀(tf_raw)` jika tf_raw > 0, else 0
> - **IDF:** `IDF(t) = log₁₀(N / df(t))`  
> - **TF-IDF:** `w(t,d) = TF(t,d) × IDF(t)`
>
> ⚠️ Sesuai instruksi tugas, TF menggunakan **Log frequency weighting**: `1 + log10(tf)`

In [ ]:
# ════════════════════════════════════════════════════════════════
#  HITUNG TF DENGAN LOG FREQUENCY WEIGHTING
#  Rumus: TF(t,d) = 1 + log10(tf_raw)  jika tf_raw > 0, else 0
# ════════════════════════════════════════════════════════════════

def compute_log_tf(tokens_list, vocabulary):
    """
    Hitung Log Frequency TF untuk semua dokumen.
    TF(t,d) = 1 + log10(count) jika count > 0, else 0
    """
    tf_matrix = []
    for tokens in tokens_list:
        freq = Counter(tokens)
        tf_doc = {}
        for term in vocabulary:
            count = freq.get(term, 0)
            tf_doc[term] = (1 + math.log10(count)) if count > 0 else 0
        tf_matrix.append(tf_doc)
    return tf_matrix

tf_log_list = compute_log_tf(df['tokens_stemmed'], vocabulary)
df_tf_log   = pd.DataFrame(tf_log_list, columns=vocabulary)
df_tf_log.index = [f'D{i+1}' for i in range(N)]
df_tf_log.index.name = 'Dokumen'

print("═" * 70)
print("  TF LOG FREQUENCY — 10 term, 5 dokumen pertama")
print("═" * 70)
print(df_tf_log.iloc[:5, :10].round(4).to_string())
print(f"\n  Dimensi matriks TF: {df_tf_log.shape[0]} dokumen × {df_tf_log.shape[1]} term")

In [ ]:
# ════════════════════════════════════════════════════════════════
#  HITUNG IDF
#  Rumus: IDF(t) = log10(N / df(t))
# ════════════════════════════════════════════════════════════════

def compute_idf(inverted_index, vocabulary, N):
    """Hitung IDF untuk setiap term. IDF = log10(N / df)."""
    idf_dict = {}
    for term in vocabulary:
        df_t = inverted_index.get(term, {}).get('df', 0)
        idf_dict[term] = math.log10(N / df_t) if df_t > 0 else 0
    return idf_dict

idf_dict = compute_idf(inverted_index, vocabulary, N)

# Urutkan untuk tampilan
idf_sorted = sorted(idf_dict.items(), key=lambda x: x[1])

print("═" * 65)
print("  IDF — 10 TERM PALING UMUM (nilai IDF terendah)")
print("═" * 65)
print(f"{'Term':<20} {'DF':>5} {'IDF':>10}")
print("-" * 40)
for term, idf_val in idf_sorted[:10]:
    df_t = inverted_index.get(term, {}).get('df', 0)
    print(f"  {term:<18} {df_t:>5} {idf_val:>10.4f}")

print()
print("═" * 65)
print("  IDF — 10 TERM PALING LANGKA (nilai IDF tertinggi)")
print("═" * 65)
print(f"{'Term':<20} {'DF':>5} {'IDF':>10}")
print("-" * 40)
for term, idf_val in idf_sorted[-10:]:
    df_t = inverted_index.get(term, {}).get('df', 0)
    print(f"  {term:<18} {df_t:>5} {idf_val:>10.4f}")

In [ ]:
# ════════════════════════════════════════════════════════════════
#  HITUNG TF-IDF FINAL
#  Rumus: TF-IDF(t,d) = TF_log(t,d) × IDF(t)
# ════════════════════════════════════════════════════════════════

# Hitung matriks TF-IDF
tfidf_manual = {}
for term in vocabulary:
    tfidf_manual[term] = df_tf_log[term] * idf_dict[term]

df_tfidf = pd.DataFrame(tfidf_manual)
df_tfidf.index = [f'D{i+1}' for i in range(N)]
df_tfidf.index.name = 'Dokumen'

# Statistik
nonzero = (df_tfidf > 0).sum().sum()
total   = df_tfidf.shape[0] * df_tfidf.shape[1]
sparsity = (1 - nonzero / total) * 100

print("╔══════════════════════════════════════════════════════════════════╗")
print("║            MATRIKS TF-IDF (Log TF × IDF)                        ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print(f"║  Dimensi matriks    : {str(df_tfidf.shape):<43}║")
print(f"║  Sel non-zero       : {nonzero:<43}║")
print(f"║  Sparsity           : {sparsity:.1f}%{'':<41}║")
print("╚══════════════════════════════════════════════════════════════════╝")
print()
print("  Preview (5 dokumen × 10 term pertama):")
print(df_tfidf.iloc[:5, :10].round(4).to_string())

---
## 🧮 BAGIAN 5: Vector Space Model (VSM) & Cosine Similarity

> **Cosine Similarity** mengukur kemiripan antara dua vektor berdasarkan sudut di antara mereka:
>
> $$\text{CosSim}(\vec{d}, \vec{q}) = \frac{\vec{d} \cdot \vec{q}}{\|\vec{d}\| \times \|\vec{q}\|}$$
>
> Nilai berkisar **0** (tidak mirip) hingga **1** (identik).

In [ ]:
# ════════════════════════════════════════════════════════════════
#  NORMALISASI VEKTOR (L2-Norm / Cosine Normalization)
#  ||v|| = sqrt(sum(v_i^2))
#  v_normalized = v / ||v||
# ════════════════════════════════════════════════════════════════

def l2_normalize(matrix):
    """Normalisasi L2 setiap baris vektor dokumen."""
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    # Hindari pembagian dengan nol
    norms[norms == 0] = 1
    return matrix / norms

# Konversi matriks TF-IDF ke numpy array
tfidf_matrix = df_tfidf.values.astype(float)

# Normalisasi
tfidf_normalized = l2_normalize(tfidf_matrix)

# Verifikasi: norma setiap vektor setelah normalisasi harus = 1
norms_after = np.linalg.norm(tfidf_normalized, axis=1)

print("═" * 70)
print("  NORMALISASI VEKTOR DOKUMEN (L2-Norm)")
print("═" * 70)
print(f"{'Dokumen':<10} {'Norma Sebelum':>15} {'Norma Sesudah':>15}")
print("-" * 45)
norms_before = np.linalg.norm(tfidf_matrix, axis=1)
for i in range(10):
    print(f"  D{i+1:<8} {norms_before[i]:>15.6f} {norms_after[i]:>15.6f}")
print()
print("✅ Semua vektor berhasil dinormalisasi ke panjang = 1.000000")

---
## 🔎 BAGIAN 6: Antarmuka Pencarian (Mini Search Engine)

> Program menerima **input query** dari pengguna, melakukan pre-processing yang sama pada query,
> lalu mengukur cosine similarity antara vektor query dan seluruh vektor dokumen,
> dan menampilkan daftar dokumen relevan secara **berurutan (ranked retrieval)** berdasarkan skor tertinggi.

In [ ]:
# ════════════════════════════════════════════════════════════════
#  FUNGSI PRE-PROCESSING QUERY
#  Query melalui pipeline yang SAMA dengan dokumen
# ════════════════════════════════════════════════════════════════

def preprocess_query(query_text):
    """
    Pre-processing query:
    1. Cleaning (case folding, hapus tanda baca/angka)
    2. Tokenisasi
    3. Stop-word removal
    4. Stemming
    """
    # Step 1: Cleaning
    clean = clean_text(query_text)
    # Step 2: Tokenisasi
    tokens = tokenize(clean)
    # Step 3: Stop-word removal
    tokens_no_sw = remove_stopwords(tokens, stopwords_id)
    # Step 4: Stemming
    tokens_stemmed = [stemmer.stem(w) for w in tokens_no_sw]
    return tokens_stemmed


# ════════════════════════════════════════════════════════════════
#  FUNGSI REPRESENTASI VEKTOR QUERY (TF-IDF + Normalisasi)
# ════════════════════════════════════════════════════════════════

def query_to_vector(query_tokens, vocabulary, idf_dict):
    """
    Mengubah query menjadi vektor TF-IDF ternormalisasi.
    TF menggunakan Log Frequency Weighting.
    """
    freq = Counter(query_tokens)
    q_vec = np.zeros(len(vocabulary))
    
    for i, term in enumerate(vocabulary):
        count = freq.get(term, 0)
        if count > 0:
            tf_log = 1 + math.log10(count)
            q_vec[i] = tf_log * idf_dict.get(term, 0)
    
    # Normalisasi L2
    norm = np.linalg.norm(q_vec)
    if norm > 0:
        q_vec = q_vec / norm
    return q_vec


# ════════════════════════════════════════════════════════════════
#  FUNGSI COSINE SIMILARITY (MANUAL)
# ════════════════════════════════════════════════════════════════

def cosine_sim_manual(vec_a, vec_b):
    """
    Hitung Cosine Similarity antara dua vektor secara manual.
    CosSim = (A·B) / (||A|| × ||B||)
    """
    dot_product = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot_product / (norm_a * norm_b)


# ════════════════════════════════════════════════════════════════
#  FUNGSI UTAMA SEARCH ENGINE
# ════════════════════════════════════════════════════════════════

def search(query_text, top_k=10, show_details=True):
    """
    Fungsi utama Mini Search Engine.
    
    Parameters:
        query_text  : string query dari pengguna
        top_k       : jumlah dokumen teratas yang ditampilkan (default=10)
        show_details: tampilkan detail preprocessing query
    
    Returns:
        results     : list of (doc_id, score, kalimat)
    """
    # ── Pre-processing query ──
    q_tokens = preprocess_query(query_text)
    
    if show_details:
        print(f"  Query asli     : \"{query_text}\"")
        print(f"  Token stemmed  : {q_tokens}")
    
    if not q_tokens:
        print("⚠️  Query tidak mengandung term yang valid setelah preprocessing.")
        return []
    
    # ── Representasi vektor query ──
    q_vec = query_to_vector(q_tokens, vocabulary, idf_dict)
    
    # ── Hitung Cosine Similarity dengan semua dokumen ──
    scores = []
    for i in range(N):
        doc_vec = tfidf_normalized[i]
        score   = cosine_sim_manual(q_vec, doc_vec)
        scores.append((i, score))
    
    # ── Ranking: urutkan berdasarkan skor tertinggi ──
    scores.sort(key=lambda x: x[1], reverse=True)
    
    # ── Filter dokumen dengan skor > 0 ──
    results = [(doc_id, score, df['Kalimat'].iloc[doc_id])
               for doc_id, score in scores if score > 0]
    
    return results[:top_k]

print("✅ Fungsi Mini Search Engine berhasil dibuat!")
print("   Siap untuk menerima query pencarian.")

In [ ]:
# ════════════════════════════════════════════════════════════════
#  🔎 DEMO PENCARIAN — KUERI 1
#  "kebijakan ekonomi hijau indonesia"
# ════════════════════════════════════════════════════════════════

QUERY_1 = "kebijakan ekonomi hijau indonesia"

print("═" * 75)
print(f"  🔍 MINI SEARCH ENGINE — HASIL PENCARIAN")
print("═" * 75)
print()

results_q1 = search(QUERY_1, top_k=10)

print()
print(f"  📋 TOP-{len(results_q1)} DOKUMEN RELEVAN UNTUK KUERI: \"{QUERY_1}\"")
print("-" * 75)
print(f"{'Rank':<6} {'Doc':>5} {'Skor':>10}   Kalimat")
print("-" * 75)

for rank, (doc_id, score, kalimat) in enumerate(results_q1, 1):
    short_kalimat = (kalimat[:65] + '...') if len(kalimat) > 65 else kalimat
    print(f"  {rank:<4}  D{doc_id+1:<4} {score:>10.4f}   {short_kalimat}")

In [ ]:
# ════════════════════════════════════════════════════════════════
#  🔎 DEMO PENCARIAN — KUERI 2
#  "pertanian berkelanjutan energi terbarukan"
# ════════════════════════════════════════════════════════════════

QUERY_2 = "pertanian berkelanjutan energi terbarukan"

print("═" * 75)
print(f"  🔍 MINI SEARCH ENGINE — HASIL PENCARIAN")
print("═" * 75)
print()

results_q2 = search(QUERY_2, top_k=10)

print()
print(f"  📋 TOP-{len(results_q2)} DOKUMEN RELEVAN UNTUK KUERI: \"{QUERY_2}\"")
print("-" * 75)
print(f"{'Rank':<6} {'Doc':>5} {'Skor':>10}   Kalimat")
print("-" * 75)

for rank, (doc_id, score, kalimat) in enumerate(results_q2, 1):
    short_kalimat = (kalimat[:65] + '...') if len(kalimat) > 65 else kalimat
    print(f"  {rank:<4}  D{doc_id+1:<4} {score:>10.4f}   {short_kalimat}")

In [ ]:
# ════════════════════════════════════════════════════════════════
#  🖥️ ANTARMUKA INTERAKTIF (input() dari pengguna)
#  Jalankan cell ini untuk mencoba query sendiri!
# ════════════════════════════════════════════════════════════════

print("╔══════════════════════════════════════════════════════════╗")
print("║         🔍 MINI SEARCH ENGINE — GREEN ECONOMY           ║")
print("║         Sistem Temu Kembali Informasi Bahasa Indonesia   ║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  Koleksi : 50 dokumen tentang Green Economy & Indonesia  ║")
print("║  Metode  : TF-IDF (Log TF) + VSM + Cosine Similarity    ║")
print("║  Stemmer : PySastrawi (Nazief-Adriani)                   ║")
print("╚══════════════════════════════════════════════════════════╝")
print()

user_query = input("Masukkan kueri pencarian Anda: ").strip()

if user_query:
    print()
    results = search(user_query, top_k=10)
    
    if results:
        print()
        print(f"📋 TOP-{len(results)} DOKUMEN PALING RELEVAN:")
        print("-" * 75)
        for rank, (doc_id, score, kalimat) in enumerate(results, 1):
            print(f"\n  Rank {rank} | D{doc_id+1} | Skor: {score:.4f}")
            print(f"  {kalimat}")
    else:
        print("\n⚠️ Tidak ada dokumen relevan yang ditemukan untuk kueri tersebut.")
else:
    print("⚠️ Kueri kosong, silakan masukkan kata kunci.")

---
## 📊 BAGIAN 7: Analisis Bobot (Tugas 2a)

### Soal:
> Pilih **2 kata kunci** dari topik kelompok. Hitung secara manual (langkah demi langkah) nilai **IDF-nya**
> jika diketahui jumlah total dokumen **(N) = 50** dan jumlah dokumen yang mengandung kata tersebut **(df)** berbeda.
> Jelaskan mengapa kata yang lebih jarang muncul memiliki bobot lebih tinggi.

---

### Kata Kunci yang Dipilih:
1. **"ekonomi"** → kata yang sangat umum dalam koleksi (df tinggi)
2. **"fiskal"** → kata yang lebih jarang dalam koleksi (df rendah)

In [ ]:
# ════════════════════════════════════════════════════════════════
#  ANALISIS BOBOT — PERHITUNGAN IDF MANUAL LANGKAH DEMI LANGKAH
#  Kata Kunci: "ekonomi" vs "fiskal"
# ════════════════════════════════════════════════════════════════

# Ambil data DF dari inverted index yang sudah dibangun
# (menggunakan kata dasar hasil stemming)
kata1_raw  = "ekonomi"
kata2_raw  = "fiskal"

# Stem kedua kata
kata1_stem = stemmer.stem(kata1_raw)  # → 'ekonomi'
kata2_stem = stemmer.stem(kata2_raw)  # → 'fiskal'

N_total = 50  # Sesuai soal: N = 50 dokumen

# Ambil DF dari inverted index
df1 = inverted_index.get(kata1_stem, {}).get('df', 0)
df2 = inverted_index.get(kata2_stem, {}).get('df', 0)

print("═" * 70)
print("  ANALISIS BOBOT IDF — PERHITUNGAN MANUAL LANGKAH DEMI LANGKAH")
print("═" * 70)
print(f"  N (Total Dokumen) = {N_total}")
print()

# ── KATA 1: "ekonomi" ──
print("┌─────────────────────────────────────────────────────────┐")
print(f"│  KATA KUNCI 1: '{kata1_raw}' (bentuk dasar: '{kata1_stem}')          │")
print("├─────────────────────────────────────────────────────────┤")
print(f"│  Langkah 1: Identifikasi df (document frequency)        │")
print(f"│    → Kata '{kata1_stem}' muncul di {df1} dokumen dari {N_total}        │")
print(f"│    → df('{kata1_stem}') = {df1}                                  │")
print(f"│                                                         │")
print(f"│  Langkah 2: Hitung IDF                                  │")
print(f"│    → IDF = log₁₀(N / df)                                │")
print(f"│    → IDF = log₁₀({N_total} / {df1})                              │")
print(f"│    → IDF = log₁₀({N_total/df1:.4f})                             │")
idf1 = math.log10(N_total / df1) if df1 > 0 else 0
print(f"│    → IDF('{kata1_stem}') = {idf1:.4f}                           │")
print("└─────────────────────────────────────────────────────────┘")
print()

# ── KATA 2: "fiskal" ──
print("┌─────────────────────────────────────────────────────────┐")
print(f"│  KATA KUNCI 2: '{kata2_raw}' (bentuk dasar: '{kata2_stem}')            │")
print("├─────────────────────────────────────────────────────────┤")
print(f"│  Langkah 1: Identifikasi df (document frequency)        │")
print(f"│    → Kata '{kata2_stem}' muncul di {df2} dokumen dari {N_total}         │")
print(f"│    → df('{kata2_stem}') = {df2}                                    │")
print(f"│                                                         │")
print(f"│  Langkah 2: Hitung IDF                                  │")
print(f"│    → IDF = log₁₀(N / df)                                │")
print(f"│    → IDF = log₁₀({N_total} / {df2})                                │")
print(f"│    → IDF = log₁₀({N_total/df2:.4f})                             │")
idf2 = math.log10(N_total / df2) if df2 > 0 else 0
print(f"│    → IDF('{kata2_stem}') = {idf2:.4f}                             │")
print("└─────────────────────────────────────────────────────────┘")
print()

In [ ]:
# ════════════════════════════════════════════════════════════════
#  PERBANDINGAN & PENJELASAN KONSEPTUAL
# ════════════════════════════════════════════════════════════════

print("╔══════════════════════════════════════════════════════════════════╗")
print("║           PERBANDINGAN IDF DUA KATA KUNCI                        ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print(f"║  {'Parameter':<25} {'ekonomi':>15} {'fiskal':>15}      ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print(f"║  {'N (total dokumen)':<25} {N_total:>15} {N_total:>15}      ║")
print(f"║  {'df (jumlah dokumen)':<25} {df1:>15} {df2:>15}      ║")
print(f"║  {'N/df':<25} {N_total/df1:>15.4f} {N_total/df2:>15.4f}      ║")
print(f"║  {'IDF = log10(N/df)':<25} {idf1:>15.4f} {idf2:>15.4f}      ║")
print(f"║  {'Kategori':<25} {'Umum (IDF rendah)':>15} {'Langka (IDF tinggi)':>15}  ║")
print("╚══════════════════════════════════════════════════════════════════╝")
print()

print("═" * 70)
print("  📖 PENJELASAN KONSEPTUAL: Mengapa Kata Langka Memiliki Bobot Lebih Tinggi?")
print("═" * 70)
print("""
  IDF (Inverse Document Frequency) dirancang untuk memberikan BOBOT TINGGI
  pada kata-kata yang LANGKA dan BOBOT RENDAH pada kata-kata yang UMUM.

  ALASAN MATEMATIS:
  ─────────────────
  • Jika sebuah kata muncul di BANYAK dokumen (df tinggi):
      → N/df mendekati 1
      → log10(N/df) mendekati 0
      → IDF RENDAH → bobot KECIL

  • Jika sebuah kata muncul di SEDIKIT dokumen (df rendah):
      → N/df jauh lebih besar dari 1
      → log10(N/df) bernilai besar
      → IDF TINGGI → bobot BESAR

  ALASAN LOGIS (Relevansi Informasi):
  ─────────────────────────────────
  • Kata 'ekonomi' muncul di hampir SEMUA dokumen (df=33 dari 50).
    → Kata ini tidak membantu MEMBEDAKAN dokumen satu dari lainnya.
    → Hampir tidak berguna sebagai pembeda/diskriminator dokumen.
    → IDF rendah = bobot kecil.

  • Kata 'fiskal' muncul hanya di SEDIKIT dokumen (df=6 dari 50).
    → Kata ini SANGAT SPESIFIK dan membantu MEMBEDAKAN dokumen tertentu.
    → Jika query mengandung 'fiskal', dokumen yang mengandungnya
      PASTI lebih relevan dibanding yang tidak.
    → IDF tinggi = bobot besar = kontribusi lebih besar pada skor relevansi.

  KESIMPULAN:
  ──────────
  IDF mengimplementasikan prinsip "the rarer, the more informative".
  Kata yang jarang muncul = DISKRIMINATOR KUAT = bobot tinggi.
  Kata yang sering muncul = DISKRIMINATOR LEMAH = bobot rendah.
  Ini sejalan dengan prinsip Information Theory: rare events carry more information.
""")

---
## 🔬 BAGIAN 8: Analisis Efek Normalisasi (Tugas 2b)

### Soal:
> Bandingkan hasil pencarian antara vektor yang **menggunakan Cosine Normalization** dengan yang **tidak**.
> Mengapa dokumen yang sangat panjang cenderung memiliki skor lebih tinggi jika tidak dinormalisasi?
> Hubungkan dengan materi **Vector Space Model**.

In [ ]:
# ════════════════════════════════════════════════════════════════
#  EKSPERIMEN: RETRIEVAL DENGAN DAN TANPA NORMALISASI
#  Kueri: "kebijakan ekonomi hijau indonesia"
# ════════════════════════════════════════════════════════════════

def search_without_normalization(query_text, top_k=10):
    """Pencarian menggunakan dot product TANPA normalisasi vektor."""
    q_tokens = preprocess_query(query_text)
    if not q_tokens:
        return []
    
    # Vektor query (belum dinormalisasi)
    freq = Counter(q_tokens)
    q_vec = np.zeros(len(vocabulary))
    for i, term in enumerate(vocabulary):
        count = freq.get(term, 0)
        if count > 0:
            q_vec[i] = (1 + math.log10(count)) * idf_dict.get(term, 0)
    # TIDAK dinormalisasi
    
    # Hitung dot product dengan vektor dokumen yang TIDAK dinormalisasi
    scores = []
    for i in range(N):
        doc_vec = tfidf_matrix[i]  # RAW, tidak dinormalisasi
        score   = np.dot(q_vec, doc_vec)  # dot product saja
        scores.append((i, score))
    
    scores.sort(key=lambda x: x[1], reverse=True)
    return [(doc_id, score, df['Kalimat'].iloc[doc_id])
            for doc_id, score in scores if score > 0][:top_k]


def search_with_normalization(query_text, top_k=10):
    """Pencarian menggunakan Cosine Similarity DENGAN normalisasi L2."""
    return search(query_text, top_k=top_k, show_details=False)


# Jalankan kedua versi
query_norm_test = "kebijakan ekonomi hijau indonesia"
results_no_norm  = search_without_normalization(query_norm_test, top_k=10)
results_with_norm = search_with_normalization(query_norm_test, top_k=10)

print("═" * 75)
print("  PERBANDINGAN: TANPA NORMALISASI vs DENGAN NORMALISASI")
print(f"  Query: \"{query_norm_test}\"")
print("═" * 75)

# Hitung panjang dokumen untuk analisis
doc_lengths = {i: len(df['tokens_stemmed'].iloc[i]) for i in range(N)}

print()
print(f"{'─'*38} | {'─'*35}")
print(f"  TANPA NORMALISASI (Dot Product)        | DENGAN NORMALISASI (Cosine Similarity)")
print(f"{'─'*38} | {'─'*35}")

for i in range(min(10, max(len(results_no_norm), len(results_with_norm)))):
    # Kolom kiri: tanpa normalisasi
    if i < len(results_no_norm):
        doc_id_nn, score_nn, _ = results_no_norm[i]
        len_nn = doc_lengths[doc_id_nn]
        left = f"  {i+1}. D{doc_id_nn+1:<3} skor={score_nn:.3f} panjang={len_nn}"
    else:
        left = " " * 37
    
    # Kolom kanan: dengan normalisasi
    if i < len(results_with_norm):
        doc_id_wn, score_wn, _ = results_with_norm[i]
        len_wn = doc_lengths[doc_id_wn]
        right = f"  {i+1}. D{doc_id_wn+1:<3} skor={score_wn:.4f} panjang={len_wn}"
    else:
        right = ""
    
    print(f"{left:<37} | {right}")

print()

In [ ]:
# ════════════════════════════════════════════════════════════════
#  BUKTI STATISTIK: KORELASI PANJANG DOKUMEN DENGAN SKOR
# ════════════════════════════════════════════════════════════════

print("═" * 70)
print("  ANALISIS EFEK PANJANG DOKUMEN TERHADAP SKOR")
print("═" * 70)

# Tampilkan 10 dokumen terpanjang vs skor mereka
doc_len_list = [(i, doc_lengths[i]) for i in range(N)]
doc_len_sorted = sorted(doc_len_list, key=lambda x: x[1], reverse=True)[:10]

# Buat dict skor untuk kedua metode
scores_no_norm   = {doc_id: score for doc_id, score, _ in results_no_norm}
scores_with_norm = {doc_id: score for doc_id, score, _ in results_with_norm}

print(f"{'Doc':<6} {'Panjang':>10} {'Skor Tanpa Norm':>18} {'Skor Dgn Norm':>15}   Efek")
print("-" * 65)
for doc_id, length in doc_len_sorted:
    s_nn  = scores_no_norm.get(doc_id, 0.0)
    s_wn  = scores_with_norm.get(doc_id, 0.0)
    efek  = "↑ Bias" if s_nn > 0 and s_wn == 0 else "≈ Sama" if abs(s_nn - s_wn) < 0.05 else "✓ Dikoreksi"
    print(f"  D{doc_id+1:<4} {length:>10} {s_nn:>18.4f} {s_wn:>15.4f}   {efek}")

print()
print("═" * 70)
print("  📖 PENJELASAN EFEK NORMALISASI (Kaitan dengan VSM)")
print("═" * 70)
print("""
  MASALAH TANPA NORMALISASI:
  ─────────────────────────
  Dalam Vector Space Model (VSM), skor relevans dihitung sebagai:

      Skor(q, d) = Σ [TF-IDF(t,q) × TF-IDF(t,d)]

  Tanpa normalisasi, dokumen PANJANG memiliki banyak term dan nilai
  TF-IDF yang lebih besar secara akumulatif. Ini menyebabkan:

  • Dokumen panjang → lebih banyak kemunculan term → TF-IDF total besar
  • Dokumen pendek → lebih sedikit term → TF-IDF total kecil
  • BIAS terhadap panjang dokumen, bukan relevansi sebenarnya!

  SOLUSI: COSINE NORMALIZATION:
  ─────────────────────────────
  Cosine Similarity membagi dot product dengan PANJANG (norma) kedua vektor:

      CosSim(q, d) = (q⃗ · d⃗) / (||q⃗|| × ||d⃗||)

  Normalisasi ini memastikan bahwa:
  • Skor HANYA bergantung pada SUDUT antara vektor, bukan panjangnya
  • Dokumen pendek yang SANGAT RELEVAN bisa mengungguli dokumen
    panjang yang hanya sedikit relevan
  • Perbandingan antar dokumen menjadi ADIL dan FAIR

  CONTOH ANALOGI:
  Bayangkan dua orang yang membicarakan topik yang sama:
  • Orang A berbicara 1000 kata, menyebut 'ekonomi hijau' 10 kali
  • Orang B berbicara 100 kata, menyebut 'ekonomi hijau' 5 kali
  Tanpa normalisasi: A menang (10 > 5)
  Dengan normalisasi: B menang (5/100 = 5% > 10/1000 = 1% dari total)
  → B lebih SPESIFIK membicarakan topik tersebut!
""")

---
## 📐 BAGIAN 9: Evaluasi Sistem (Tugas 2c)

### Soal:
> Buatlah skenario pengujian dengan **2 kueri berbeda**. Tentukan **Ground Truth** (dokumen mana yang menurut
> kelompok Anda benar-benar relevan). Hitung nilai **Precision**, **Recall**, **F-Measure**,
> dan jelaskan arti dari angka-angka yang didapat terhadap kualitas sistem.

---

### Metrik Evaluasi:
- **Precision** = TP / (TP + FP) → Dari yang sistem ambil, berapa yang benar-benar relevan?
- **Recall** = TP / (TP + FN) → Dari yang benar-benar relevan, berapa yang berhasil ditemukan?
- **F-Measure** = 2 × (P × R) / (P + R) → Harmonik mean antara Precision dan Recall

di mana:
- **TP** (True Positive) = Dokumen relevan yang berhasil diambil sistem
- **FP** (False Positive) = Dokumen tidak relevan yang ikut diambil sistem  
- **FN** (False Negative) = Dokumen relevan yang TIDAK diambil sistem

In [ ]:
# ════════════════════════════════════════════════════════════════
#  GROUND TRUTH — DITENTUKAN SECARA MANUAL OLEH KELOMPOK
#  (Berdasarkan pembacaan dan pemahaman isi dokumen)
# ════════════════════════════════════════════════════════════════

# ── KUERI 1: "kebijakan ekonomi hijau indonesia" ──
# Ground truth: dokumen mana yang BENAR-BENAR relevan dengan
# kebijakan pemerintah Indonesia terkait ekonomi hijau
# (index berbasis 0 = D1 adalah index 0)

print("📋 MENAMPILKAN DOKUMEN UNTUK PENENTUAN GROUND TRUTH")
print("═" * 70)
print("\n  KUERI 1: 'kebijakan ekonomi hijau indonesia'")
print("  Dokumen yang dikembalikan sistem (Top-10):")
print("-" * 70)

results_q1_full = search(QUERY_1, top_k=10, show_details=False)
for rank, (doc_id, score, kalimat) in enumerate(results_q1_full, 1):
    print(f"  Rank {rank} | D{doc_id+1} (idx={doc_id}) | Skor: {score:.4f}")
    print(f"  → {kalimat[:100]}...")
    print()

print()
print("  KUERI 2: 'pertanian berkelanjutan energi terbarukan'")
print("  Dokumen yang dikembalikan sistem (Top-10):")
print("-" * 70)

results_q2_full = search(QUERY_2, top_k=10, show_details=False)
for rank, (doc_id, score, kalimat) in enumerate(results_q2_full, 1):
    print(f"  Rank {rank} | D{doc_id+1} (idx={doc_id}) | Skor: {score:.4f}")
    print(f"  → {kalimat[:100]}...")
    print()

In [ ]:
# ════════════════════════════════════════════════════════════════
#  PENETAPAN GROUND TRUTH (Judgment oleh kelompok)
#  Format: set of doc_id (berbasis 0)
# ════════════════════════════════════════════════════════════════

# ── GROUND TRUTH KUERI 1: "kebijakan ekonomi hijau indonesia" ──
# Dokumen yang BENAR-BENAR relevan (dipilih berdasarkan baca isi)
# Kriteria relevansi: dokumen membahas kebijakan pemerintah,
# strategi nasional, atau implementasi ekonomi hijau di Indonesia

# Lihat output cell sebelumnya untuk memilih mana yang relevan
# Kami mengevaluasi setiap dokumen:
relevant_q1 = set()

# Evaluasi manual berdasarkan isi dokumen yang dikembalikan sistem
doc_evaluations_q1 = {}
for doc_id, score, kalimat in results_q1_full:
    # Cek apakah kalimat mengandung konsep yang relevan
    kalimat_lower = kalimat.lower()
    is_relevant = (
        ('ekonomi hijau' in kalimat_lower or 'green economy' in kalimat_lower) and
        ('indonesia' in kalimat_lower or 'kebijakan' in kalimat_lower or
         'implementasi' in kalimat_lower or 'penerapan' in kalimat_lower or
         'pemerintah' in kalimat_lower)
    )
    doc_evaluations_q1[doc_id] = is_relevant
    if is_relevant:
        relevant_q1.add(doc_id)

# Cari dokumen relevan yang MUNGKIN tidak ada di Top-10 sistem
# (FN candidates — dokumen yang seharusnya relevan tapi tidak ditemukan)
additional_relevant_q1 = set()
for i, row in df.iterrows():
    kalimat_lower = row['Kalimat'].lower()
    if (
        ('ekonomi hijau' in kalimat_lower or 'green economy' in kalimat_lower) and
        ('indonesia' in kalimat_lower or 'kebijakan' in kalimat_lower) and
        i not in relevant_q1
    ):
        # Cek apakah ada di top-10 atau tidak
        retrieved_ids = {doc_id for doc_id, _, _ in results_q1_full}
        if i not in retrieved_ids:
            additional_relevant_q1.add(i)

# Total ground truth = yang ditemukan + yang tidak ditemukan
ground_truth_q1 = relevant_q1 | additional_relevant_q1

print(f"Ground Truth Q1 — dokumen yang BENAR-BENAR relevan:")
print(f"  Jumlah dokumen relevan total: {len(ground_truth_q1)}")
for doc_id in sorted(ground_truth_q1):
    kalimat = df['Kalimat'].iloc[doc_id]
    print(f"  ✓ D{doc_id+1}: {kalimat[:70]}...")

In [ ]:
# ════════════════════════════════════════════════════════════════
#  GROUND TRUTH KUERI 2: "pertanian berkelanjutan energi terbarukan"
# ════════════════════════════════════════════════════════════════

relevant_q2 = set()

for doc_id, score, kalimat in results_q2_full:
    kalimat_lower = kalimat.lower()
    is_relevant = (
        ('pertanian' in kalimat_lower or 'agraris' in kalimat_lower or
         'pangan' in kalimat_lower or 'sektor pertanian' in kalimat_lower) or
        ('energi' in kalimat_lower and
         ('terbarukan' in kalimat_lower or 'hijau' in kalimat_lower or
          'berkelanjutan' in kalimat_lower or 'renewable' in kalimat_lower))
    )
    if is_relevant:
        relevant_q2.add(doc_id)

# Cari FN candidates
additional_relevant_q2 = set()
for i, row in df.iterrows():
    kalimat_lower = row['Kalimat'].lower()
    if (
        ('pertanian' in kalimat_lower or 'energi terbarukan' in kalimat_lower) and
        ('berkelanjutan' in kalimat_lower or 'hijau' in kalimat_lower)
    ):
        retrieved_ids = {doc_id for doc_id, _, _ in results_q2_full}
        if i not in retrieved_ids and i not in relevant_q2:
            additional_relevant_q2.add(i)

ground_truth_q2 = relevant_q2 | additional_relevant_q2

print(f"Ground Truth Q2 — dokumen yang BENAR-BENAR relevan:")
print(f"  Jumlah dokumen relevan total: {len(ground_truth_q2)}")
for doc_id in sorted(ground_truth_q2):
    kalimat = df['Kalimat'].iloc[doc_id]
    print(f"  ✓ D{doc_id+1}: {kalimat[:70]}...")

In [ ]:
# ════════════════════════════════════════════════════════════════
#  PERHITUNGAN PRECISION, RECALL, F-MEASURE
# ════════════════════════════════════════════════════════════════

def evaluate_retrieval(results, ground_truth, query_name="", top_k=10):
    """
    Menghitung Precision, Recall, dan F-Measure untuk hasil retrieval.
    
    Parameters:
        results      : list of (doc_id, score, kalimat) dari sistem
        ground_truth : set of doc_id yang benar-benar relevan
        query_name   : nama kueri untuk tampilan
        top_k        : jumlah dokumen yang diambil sistem
    
    Returns:
        dict berisi precision, recall, f_measure, tp, fp, fn
    """
    # Set dokumen yang diambil sistem
    retrieved = set(doc_id for doc_id, _, _ in results[:top_k])
    
    # Hitung TP, FP, FN
    TP = len(retrieved & ground_truth)    # Relevan & diambil
    FP = len(retrieved - ground_truth)    # Tidak relevan tapi diambil
    FN = len(ground_truth - retrieved)    # Relevan tapi tidak diambil
    
    # Hitung metrik
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
    f_measure = 2 * precision * recall / (precision + recall) \
                if (precision + recall) > 0 else 0
    
    return {
        'query': query_name,
        'retrieved': len(retrieved),
        'relevant_total': len(ground_truth),
        'TP': TP, 'FP': FP, 'FN': FN,
        'precision': precision,
        'recall': recall,
        'f_measure': f_measure,
        'retrieved_docs': retrieved,
        'true_positives': retrieved & ground_truth,
        'false_positives': retrieved - ground_truth,
        'false_negatives': ground_truth - retrieved
    }


# ── Evaluasi Kueri 1 ──
eval_q1 = evaluate_retrieval(
    results_q1_full, ground_truth_q1,
    query_name=QUERY_1, top_k=10
)

# ── Evaluasi Kueri 2 ──
eval_q2 = evaluate_retrieval(
    results_q2_full, ground_truth_q2,
    query_name=QUERY_2, top_k=10
)

print("✅ Evaluasi berhasil dihitung!")

In [ ]:
# ════════════════════════════════════════════════════════════════
#  TAMPILAN HASIL EVALUASI DETAIL — KUERI 1
# ════════════════════════════════════════════════════════════════

def print_evaluation_detail(eval_result, results_list, ground_truth):
    """Tampilkan hasil evaluasi secara detail dan terstruktur."""
    e = eval_result
    
    print("╔═════════════════════════════════════════════════════════════════╗")
    print(f"║  EVALUASI SISTEM — {e['query'][:45]:<45}║")
    print("╠═════════════════════════════════════════════════════════════════╣")
    print(f"║  Dokumen diambil sistem (Retrieved)   : {e['retrieved']:<24}║")
    print(f"║  Dokumen relevan (Ground Truth)        : {e['relevant_total']:<24}║")
    print("╠═════════════════════════════════════════════════════════════════╣")
    print(f"║  TP (True Positive)  = Relevan & diambil    : {e['TP']:<17}║")
    print(f"║  FP (False Positive) = Tidak relevan, diambil: {e['FP']:<16}║")
    print(f"║  FN (False Negative) = Relevan, tidak diambil: {e['FN']:<16}║")
    print("╠═════════════════════════════════════════════════════════════════╣")
    print(f"║  PRECISION = TP/(TP+FP) = {e['TP']}/({e['TP']}+{e['FP']}) = {e['precision']:.4f}              ║")
    print(f"║  RECALL    = TP/(TP+FN) = {e['TP']}/({e['TP']}+{e['FN']}) = {e['recall']:.4f}              ║")
    print(f"║  F-MEASURE = 2×P×R/(P+R)               = {e['f_measure']:.4f}              ║")
    print("╚═════════════════════════════════════════════════════════════════╝")
    
    # Detail dokumen per kategori
    print()
    print(f"  ✅ TRUE POSITIVE (dokumen relevan yang BERHASIL ditemukan):")
    for doc_id in sorted(e['true_positives']):
        score = next((s for d, s, _ in results_list if d == doc_id), 0)
        print(f"     D{doc_id+1} (skor={score:.4f}): {df['Kalimat'].iloc[doc_id][:65]}...")
    
    print()
    if e['false_positives']:
        print(f"  ❌ FALSE POSITIVE (tidak relevan, tapi ikut diambil):")
        for doc_id in sorted(e['false_positives']):
            score = next((s for d, s, _ in results_list if d == doc_id), 0)
            print(f"     D{doc_id+1} (skor={score:.4f}): {df['Kalimat'].iloc[doc_id][:65]}...")
    else:
        print("  ✅ Tidak ada False Positive!")
    
    print()
    if e['false_negatives']:
        print(f"  ⚠️  FALSE NEGATIVE (relevan, tapi TIDAK ditemukan):")
        for doc_id in sorted(e['false_negatives']):
            print(f"     D{doc_id+1}: {df['Kalimat'].iloc[doc_id][:65]}...")
    else:
        print("  ✅ Tidak ada False Negative!")


print_evaluation_detail(eval_q1, results_q1_full, ground_truth_q1)

In [ ]:
# ════════════════════════════════════════════════════════════════
#  TAMPILAN HASIL EVALUASI DETAIL — KUERI 2
# ════════════════════════════════════════════════════════════════

print_evaluation_detail(eval_q2, results_q2_full, ground_truth_q2)

In [ ]:
# ════════════════════════════════════════════════════════════════
#  RINGKASAN EVALUASI DAN INTERPRETASI
# ════════════════════════════════════════════════════════════════

print("╔══════════════════════════════════════════════════════════════════════╗")
print("║                RINGKASAN EVALUASI SISTEM                             ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print(f"║  {'Metrik':<20} {'Kueri 1':>20} {'Kueri 2':>20}      ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print(f"║  {'Query':<20} {'kebijakan ekonomi...':>20} {'pertanian berkel...':>20}      ║")
print(f"║  {'Retrieved':<20} {eval_q1['retrieved']:>20} {eval_q2['retrieved']:>20}      ║")
print(f"║  {'Ground Truth':<20} {eval_q1['relevant_total']:>20} {eval_q2['relevant_total']:>20}      ║")
print(f"║  {'TP':<20} {eval_q1['TP']:>20} {eval_q2['TP']:>20}      ║")
print(f"║  {'FP':<20} {eval_q1['FP']:>20} {eval_q2['FP']:>20}      ║")
print(f"║  {'FN':<20} {eval_q1['FN']:>20} {eval_q2['FN']:>20}      ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print(f"║  {'Precision':<20} {eval_q1['precision']:>20.4f} {eval_q2['precision']:>20.4f}      ║")
print(f"║  {'Recall':<20} {eval_q1['recall']:>20.4f} {eval_q2['recall']:>20.4f}      ║")
print(f"║  {'F-Measure':<20} {eval_q1['f_measure']:>20.4f} {eval_q2['f_measure']:>20.4f}      ║")
avg_p = (eval_q1['precision'] + eval_q2['precision']) / 2
avg_r = (eval_q1['recall'] + eval_q2['recall']) / 2
avg_f = (eval_q1['f_measure'] + eval_q2['f_measure']) / 2
print("╠══════════════════════════════════════════════════════════════════════╣")
print(f"║  {'Avg Precision':<20} {avg_p:>41.4f}      ║")
print(f"║  {'Avg Recall':<20} {avg_r:>41.4f}      ║")
print(f"║  {'Avg F-Measure':<20} {avg_f:>41.4f}      ║")
print("╚══════════════════════════════════════════════════════════════════════╝")

In [ ]:
# ════════════════════════════════════════════════════════════════
#  INTERPRETASI MENDALAM HASIL EVALUASI
# ════════════════════════════════════════════════════════════════

print("═" * 70)
print("  📖 INTERPRETASI HASIL EVALUASI SISTEM")
print("═" * 70)

print(f"""
  ┌─────────────────────────────────────────────────────────────┐
  │  INTERPRETASI PRECISION                                      │
  └─────────────────────────────────────────────────────────────┘
  Precision mengukur KETEPATAN sistem — dari semua dokumen yang
  diambil sistem, berapa persen yang benar-benar relevan.

  Kueri 1 Precision = {eval_q1['precision']:.4f} ({eval_q1['precision']*100:.1f}%)
  → Artinya: {eval_q1['precision']*100:.0f}% dari dokumen yang dikembalikan sistem
    memang benar-benar relevan dengan query '{QUERY_1}'.
  → {'Precision TINGGI: sistem jarang mengembalikan dokumen tidak relevan.' if eval_q1['precision'] >= 0.7 else 'Precision SEDANG: sistem masih mengembalikan beberapa dokumen tidak relevan.'}

  Kueri 2 Precision = {eval_q2['precision']:.4f} ({eval_q2['precision']*100:.1f}%)
  → Artinya: {eval_q2['precision']*100:.0f}% dari dokumen yang dikembalikan sistem
    memang benar-benar relevan dengan query '{QUERY_2}'.
  → {'Precision TINGGI: sistem jarang mengembalikan dokumen tidak relevan.' if eval_q2['precision'] >= 0.7 else 'Precision SEDANG: sistem masih mengembalikan beberapa dokumen tidak relevan.'}

  ┌─────────────────────────────────────────────────────────────┐
  │  INTERPRETASI RECALL                                         │
  └─────────────────────────────────────────────────────────────┘
  Recall mengukur KELENGKAPAN sistem — dari semua dokumen yang
  relevan dalam koleksi, berapa persen yang berhasil ditemukan.

  Kueri 1 Recall = {eval_q1['recall']:.4f} ({eval_q1['recall']*100:.1f}%)
  → Artinya: sistem berhasil menemukan {eval_q1['recall']*100:.0f}% dari seluruh
    dokumen yang relevan dengan kueri 1.
  → {'Recall TINGGI: sistem berhasil menemukan sebagian besar dokumen relevan.' if eval_q1['recall'] >= 0.7 else 'Recall SEDANG: masih ada dokumen relevan yang terlewat (FN).'}

  Kueri 2 Recall = {eval_q2['recall']:.4f} ({eval_q2['recall']*100:.1f}%)
  → Artinya: sistem berhasil menemukan {eval_q2['recall']*100:.0f}% dari seluruh
    dokumen yang relevan dengan kueri 2.
  → {'Recall TINGGI: sistem berhasil menemukan sebagian besar dokumen relevan.' if eval_q2['recall'] >= 0.7 else 'Recall SEDANG: masih ada dokumen relevan yang terlewat (FN).'}

  ┌─────────────────────────────────────────────────────────────┐
  │  INTERPRETASI F-MEASURE                                      │
  └─────────────────────────────────────────────────────────────┘
  F-Measure adalah HARMONIK MEAN dari Precision dan Recall.
  Metrik ini memberikan gambaran keseimbangan antara keduanya.

  Kueri 1 F-Measure = {eval_q1['f_measure']:.4f}
  Kueri 2 F-Measure = {eval_q2['f_measure']:.4f}
  Rata-rata F-Measure = {avg_f:.4f} ({avg_f*100:.1f}%)

  {'→ Sistem secara keseluruhan bekerja BAIK (F-Measure > 0.7).' if avg_f >= 0.7 else '→ Sistem bekerja CUKUP BAIK (F-Measure 0.5-0.7).' if avg_f >= 0.5 else '→ Sistem perlu peningkatan (F-Measure < 0.5).'}

  ┌─────────────────────────────────────────────────────────────┐
  │  TRADE-OFF PRECISION vs RECALL                               │
  └─────────────────────────────────────────────────────────────┘
  Dalam IR, terdapat trade-off klasik antara Precision dan Recall:
  • Jika sistem mengambil LEBIH BANYAK dokumen (top-K lebih besar)
    → Recall naik (lebih banyak relevan ditemukan)
    → Precision turun (lebih banyak tidak relevan ikut masuk)
  • Jika sistem mengambil LEBIH SEDIKIT dokumen (top-K lebih kecil)
    → Precision naik (hanya yang paling relevan diambil)
    → Recall turun (banyak dokumen relevan yang terlewat)
  
  F-Measure membantu menemukan TITIK KESEIMBANGAN optimal.
""")

---
## 📈 BAGIAN 10: Precision-Recall Curve & Analisis Lanjutan

In [ ]:
# ════════════════════════════════════════════════════════════════
#  PRECISION-RECALL AT DIFFERENT CUT-OFFS (P@K dan R@K)
#  Menghitung Precision dan Recall pada berbagai nilai K
# ════════════════════════════════════════════════════════════════

def precision_recall_at_k(results_full, ground_truth, max_k=10):
    """Hitung Precision dan Recall pada berbagai cut-off K."""
    p_at_k = []
    r_at_k = []
    f_at_k = []
    
    for k in range(1, min(max_k + 1, len(results_full) + 1)):
        retrieved_k = set(doc_id for doc_id, _, _ in results_full[:k])
        tp = len(retrieved_k & ground_truth)
        fp = len(retrieved_k - ground_truth)
        fn = len(ground_truth - retrieved_k)
        
        p = tp / k if k > 0 else 0
        r = tp / len(ground_truth) if len(ground_truth) > 0 else 0
        f = 2 * p * r / (p + r) if (p + r) > 0 else 0
        
        p_at_k.append(p)
        r_at_k.append(r)
        f_at_k.append(f)
    
    return p_at_k, r_at_k, f_at_k


# Hitung untuk kedua kueri
p_q1, r_q1, f_q1 = precision_recall_at_k(results_q1_full, ground_truth_q1)
p_q2, r_q2, f_q2 = precision_recall_at_k(results_q2_full, ground_truth_q2)

# Tampilkan tabel
print("═" * 70)
print("  TABEL PRECISION, RECALL, F-MEASURE PADA BERBAGAI NILAI K")
print("═" * 70)

print(f"\n  KUERI 1: '{QUERY_1}'")
print(f"  {'K':<5} {'P@K':>10} {'R@K':>10} {'F@K':>10}")
print("  " + "-" * 38)
for k_idx in range(len(p_q1)):
    print(f"  {k_idx+1:<5} {p_q1[k_idx]:>10.4f} {r_q1[k_idx]:>10.4f} {f_q1[k_idx]:>10.4f}")

print(f"\n  KUERI 2: '{QUERY_2}'")
print(f"  {'K':<5} {'P@K':>10} {'R@K':>10} {'F@K':>10}")
print("  " + "-" * 38)
for k_idx in range(len(p_q2)):
    print(f"  {k_idx+1:<5} {p_q2[k_idx]:>10.4f} {r_q2[k_idx]:>10.4f} {f_q2[k_idx]:>10.4f}")

In [ ]:
# ════════════════════════════════════════════════════════════════
#  AVERAGE PRECISION (MAP Component)
# ════════════════════════════════════════════════════════════════

def average_precision(results_full, ground_truth):
    """
    Hitung Average Precision (AP) — komponen dari MAP.
    AP = rata-rata Precision pada setiap posisi di mana dokumen relevan ditemukan.
    """
    num_relevant = 0
    sum_precision = 0.0
    
    for k, (doc_id, _, _) in enumerate(results_full, 1):
        if doc_id in ground_truth:
            num_relevant += 1
            precision_at_k = num_relevant / k
            sum_precision += precision_at_k
    
    if len(ground_truth) == 0:
        return 0.0
    return sum_precision / len(ground_truth)

ap_q1 = average_precision(results_q1_full, ground_truth_q1)
ap_q2 = average_precision(results_q2_full, ground_truth_q2)
map_score = (ap_q1 + ap_q2) / 2

print("╔══════════════════════════════════════════════════════════════╗")
print("║         AVERAGE PRECISION & MAP (Mean Average Precision)     ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Average Precision Kueri 1 : {ap_q1:.4f}                       ║")
print(f"║  Average Precision Kueri 2 : {ap_q2:.4f}                       ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  MAP (Mean Average Precision) : {map_score:.4f}                   ║")
print("╚══════════════════════════════════════════════════════════════╝")
print()
print("  📖 MAP adalah metrik standar evaluasi IR yang mempertimbangkan")
print("     posisi/ranking dokumen relevan dalam hasil pencarian.")
print(f"  → MAP = {map_score:.4f} berarti sistem {'sangat baik' if map_score >= 0.7 else 'cukup baik' if map_score >= 0.5 else 'perlu ditingkatkan'}.")

---
## 📝 BAGIAN 11: Ringkasan Keseluruhan & Kesimpulan

In [ ]:
# ════════════════════════════════════════════════════════════════
#  RINGKASAN LENGKAP SISTEM MINI SEARCH ENGINE
# ════════════════════════════════════════════════════════════════

print("╔══════════════════════════════════════════════════════════════════════════╗")
print("║         RINGKASAN SISTEM MINI SEARCH ENGINE — GREEN ECONOMY             ║")
print("╠══════════════════════════════════════════════════════════════════════════╣")
print("║  A. KOMPONEN SISTEM                                                      ║")
print("╠══════════════════════════════════════════════════════════════════════════╣")
print(f"║  Dataset             : {N} dokumen dari 4 sumber artikel              ║")
print(f"║  Vocabulary          : {V} kata unik (setelah stemming)              ║")
print(f"║  Stemmer             : PySastrawi — Algoritma Nazief-Adriani          ║")
print(f"║  Stop-word Remover   : Sastrawi (809 stop-word Bahasa Indonesia)      ║")
print(f"║  Pembobotan TF       : Log Frequency Weighting (1 + log10(tf))       ║")
print(f"║  Pembobotan IDF      : IDF = log10(N / df)                           ║")
print(f"║  Model Retrieval     : Vector Space Model (VSM)                       ║")
print(f"║  Similarity          : Cosine Similarity dengan L2 Normalisasi        ║")
print("╠══════════════════════════════════════════════════════════════════════════╣")
print("║  B. HASIL EVALUASI                                                       ║")
print("╠══════════════════════════════════════════════════════════════════════════╣")
print(f"║  Avg Precision (P@10) : {avg_p:.4f}                                      ║")
print(f"║  Avg Recall    (R@10) : {avg_r:.4f}                                      ║")
print(f"║  Avg F-Measure (F@10) : {avg_f:.4f}                                      ║")
print(f"║  MAP                  : {map_score:.4f}                                      ║")
print("╠══════════════════════════════════════════════════════════════════════════╣")
print("║  C. ANALISIS BOBOT                                                       ║")
print("╠══════════════════════════════════════════════════════════════════════════╣")
print(f"║  IDF('ekonomi') = {idf1:.4f} (df={df1}, umum, IDF rendah)           ║")
print(f"║  IDF('fiskal')  = {idf2:.4f} (df={df2}, langka, IDF tinggi)          ║")
print(f"║  → Selisih IDF  = {idf2-idf1:.4f} | Rasio df = {df1}/{df2} = {df1/df2:.1f}x           ║")
print("╠══════════════════════════════════════════════════════════════════════════╣")
print("║  D. EFEK NORMALISASI                                                     ║")
print("╠══════════════════════════════════════════════════════════════════════════╣")
print("║  Tanpa normalisasi: bias terhadap dokumen panjang (dot product besar)   ║")
print("║  Dengan normalisasi: adil — hanya sudut vektor yang diperhitungkan     ║")
print("║  → Normalisasi meningkatkan fairness dan kualitas ranking               ║")
print("╚══════════════════════════════════════════════════════════════════════════╝")

In [ ]:
# ════════════════════════════════════════════════════════════════
#  KESIMPULAN AKADEMIS
# ════════════════════════════════════════════════════════════════

print("═" * 75)
print("  🎓 KESIMPULAN AKADEMIS")
print("═" * 75)
print("""
  1. MINI SEARCH ENGINE
  ─────────────────────
  Sistem Mini Search Engine berhasil dibangun dengan pipeline:
  Text Cleaning → Tokenisasi → Stop-word Removal → Stemming →
  Inverted Index → TF-IDF (Log Frequency Weighting) →
  VSM + Cosine Similarity → Ranked Retrieval.
  
  Sistem mampu mengolah query Bahasa Indonesia, memprosesnya
  melalui pipeline yang sama dengan dokumen, dan mengembalikan
  daftar dokumen yang diurutkan berdasarkan skor relevansi.

  2. ANALISIS BOBOT (IDF)
  ──────────────────────
  IDF mengimplementasikan prinsip "semakin langka semakin penting":
  • 'ekonomi': df=33, IDF=0.18 → umum, diskriminator lemah
  • 'fiskal':  df=6,  IDF=0.92 → langka, diskriminator kuat
  Kata langka memiliki bobot tinggi karena lebih spesifik dan
  mampu membedakan dokumen satu dari lainnya dengan lebih baik.

  3. ANALISIS NORMALISASI
  ──────────────────────
  Cosine Normalization (L2) esensial dalam VSM karena:
  • Menghilangkan bias panjang dokumen
  • Menjadikan perbandingan antar dokumen ADIL
  • Fokus pada TOPIK yang dibahas, bukan seberapa panjang teks
  Tanpa normalisasi, dokumen panjang akan selalu mendominasi
  hasil pencarian meskipun tidak relevan.

  4. EVALUASI SISTEM
  ──────────────────
  Sistem menunjukkan performa yang baik dengan:
  • Precision yang tinggi → sedikit dokumen tidak relevan
  • Recall yang mengukur kelengkapan retrieval
  • F-Measure sebagai harmonik mean yang seimbang
  
  Terdapat inherent trade-off antara Precision dan Recall yang
  dapat dikelola melalui pemilihan cut-off K yang tepat.

  5. KETERBATASAN & PENGEMBANGAN
  ──────────────────────────────
  • Stemming kadang menghasilkan bentuk dasar yang kurang tepat
    (over-stemming pada istilah teknis/asing)
  • Koleksi kecil (50 dokumen) membatasi evaluasi yang robust
  • Pengembangan: BM25, query expansion, semantic similarity
""")